In [1]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, average_precision_score, classification_report

In [2]:
def load_data(path="../data/transformer_data.csv"):
    df = pd.read_csv(path)
    return df


In [3]:
def train_model(df):
    feature_cols = [
        "age_years", "load_factor", "maintenance_score",
        "oil_quality_index", "temperature_rise_c"
    ]
    X = df[feature_cols].values
    y = df["failure_within_1yr"].values

    # Held-out split so the reported metrics reflect generalization, not
    # memorization of the training rows.
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )

    train_scaler = StandardScaler()
    X_train_scaled = train_scaler.fit_transform(X_train)
    X_test_scaled = train_scaler.transform(X_test)

    eval_model = RandomForestClassifier(random_state=42)
    eval_model.fit(X_train_scaled, y_train)  # <-- train the eval model

    y_test_scores = eval_model.predict_proba(X_test_scaled)[:, 1]
    y_test_pred = eval_model.predict(X_test_scaled)

    print("Held-out ROC-AUC:", roc_auc_score(y_test, y_test_scores))
    print("Held-out Average Precision:", average_precision_score(y_test, y_test_scores))
    print(classification_report(y_test, y_test_pred))

    # Refit on the full dataset for deployment scoring, now that the
    # held-out numbers above give an honest read on generalization.
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    model = RandomForestClassifier(random_state=42)
    model.fit(X_scaled, y)

    y_scores = model.predict_proba(X_scaled)[:, 1]
    y_pred = model.predict(X_scaled)

    return model, scaler, feature_cols, y_scores, y_pred

In [4]:
def score_all_transformers(df, y_scores, y_pred):
    result = df.copy()
    result["failure_score"] = y_scores
    result["predicted_failure"] = y_pred
    result["tier"] = pd.qcut(result["failure_score"], q=3, labels=["low", "medium", "high"])
    result.to_csv("../data/transformer_failure_scores.csv", index=False)  # <-- fixed path
    return result.sort_values("failure_score", ascending=False)


In [5]:
df = load_data()
model, scaler, feature_cols, y_scores, y_pred = train_model(df)
scored = score_all_transformers(df, y_scores, y_pred)

scored.head()

Held-out ROC-AUC: 0.8000541125541125
Held-out Average Precision: 0.5032233552965907
              precision    recall  f1-score   support

           0       0.87      0.94      0.90       132
           1       0.53      0.32      0.40        28

    accuracy                           0.83       160
   macro avg       0.70      0.63      0.65       160
weighted avg       0.81      0.83      0.81       160



,transformer_id,age_years,load_factor,maintenance_score,oil_quality_index,temperature_rise_c,failure_within_1yr,failure_score,predicted_failure,tier
123,T0124,35.0,0.473,0.364,0.473,65.6,1,0.98,1,high
722,T0723,28.8,0.856,0.384,0.330,77.0,1,0.98,1,high
705,T0706,35.0,0.595,0.354,0.439,74.3,1,0.97,1,high
369,T0370,15.3,0.699,0.251,0.473,67.5,1,0.95,1,high
304,T0305,25.2,0.601,0.200,0.316,67.6,1,0.94,1,high
